In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

# loading goemotions dataset

dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
train_df= pd.DataFrame(dataset['train'])
test_df= pd.DataFrame(dataset['test'])


label_names = dataset['train'].features['labels'].feature.names

train_single = train_df[train_df['labels'].apply(len)==1].copy()
test_single = test_df[test_df['labels'].apply(len)==1].copy()

train_single['label']= train_single['labels'].apply(lambda x : x[0])
test_single['label']= test_single['labels'].apply(lambda x : x[0])

train_single['emotion'] = train_single['label'].apply(lambda x : label_names[x])
test_single['emotion']= test_single['label'].apply(lambda x : label_names[x])

print(f"Training examples : {len(train_single)}")
print(f"Test examples : {len(test_single)}")

Training examples : 36308
Test examples : 4590


In [2]:
X_train = train_single['text'].tolist()
Y_train = train_single['label'].tolist()

X_test = test_single['text'].tolist()
Y_test = test_single['label'].tolist()

print(f"X_train type : {type(X_train)}")
print(f"First training text : {X_train[0]}")
print(f"First training label : {Y_train[0]} = {label_names[Y_train[0]]}")


X_train type : <class 'list'>
First training text : My favourite food is anything I didn't have to cook myself.
First training label : 27 = neutral


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# max_features=10000 means: only keep the 10,000 most important words
# This prevents the model from getting overwhelmed by rare typos
vectorizer = TfidfVectorizer(max_features=10000)

# fit_transform: LEARN the vocabulary from training data AND convert to numbers

X_train_tfidf = vectorizer.fit_transform(X_train)

# transform only: use the ALREADY LEARNED vocabulary, just convert

X_test_tfidf = vectorizer.transform(X_test)

print(f"Training matrix shape : {X_train_tfidf.shape}")
print(f"Test matrix shape : {X_test_tfidf.shape}")




Training matrix shape : (36308, 10000)
Test matrix shape : (4590, 10000)


In [4]:
# What words did TF-IDF learn?
# Let's see the vocabulary

feature_names = vectorizer.get_feature_names_out()
print(f"Total vocabulary size: {len(feature_names)}")
print(f"First 20 words: {feature_names[:20]}")
print(f"Last 20 words: {feature_names[-20:]}")

Total vocabulary size: 10000
First 20 words: ['00' '000' '01' '05' '06' '10' '100' '1000' '100k' '101' '10k' '10th'
 '10x' '11' '11am' '11th' '12' '120' '1200' '125']
Last 20 words: ['yum' 'yummy' 'yup' 'zealand' 'zebra' 'zebras' 'zelda' 'zero' 'ziip'
 'zim' 'zinger' 'zion' 'zipper' 'zippers' 'zombie' 'zombies' 'zone'
 'zones' 'zoo' 'zoom']


In [5]:
tfidf_sum = X_train_tfidf.sum(axis=0)
tfidf_array = np.array(tfidf_sum).flatten()
top_indices = tfidf_array.argsort()[-20:][::-1]
top_words =[(feature_names[i] , tfidf_array[i]) for i in top_indices]

print("Top 20 most distinctive words in GoEmotions : ")
for word , score in top_words:
    print(f" {word}: {score:.3f}")
    

Top 20 most distinctive words in GoEmotions : 
 the: 1521.992
 you: 1295.334
 to: 1185.271
 it: 1169.485
 name: 1140.355
 that: 1110.643
 is: 998.295
 and: 895.464
 this: 888.133
 of: 798.219
 for: 732.016
 in: 699.938
 was: 579.660
 my: 559.190
 not: 553.998
 like: 536.221
 so: 530.266
 he: 519.956
 be: 512.372
 but: 501.884


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_v2= TfidfVectorizer(max_features = 10000 , stop_words ='english' )

X_train_tfidf_v2= vectorizer_v2.fit_transform(X_train)
X_test_tfidf_v2= vectorizer_v2.transform(X_test)

print(f"Training matrix shape :{X_train_tfidf_v2.shape}")
print(f"Testing matrix shape :{X_test_tfidf_v2.shape}")

Training matrix shape :(36308, 10000)
Testing matrix shape :(4590, 10000)


In [7]:
feature_names_v2 = vectorizer_v2.get_feature_names_out()
tfidf_sum_v2 = X_train_tfidf_v2.sum(axis=0)
tfidf_array_v2 = np.array(tfidf_sum_v2).flatten()
top_indices_v2 = tfidf_array_v2.argsort()[-20:][::-1]
top_words_v2 = [(feature_names_v2[i] , tfidf_array_v2[i]) for i in top_indices_v2]

print("Top 20 most distinctive words (Without stopwords) : ")
for word , score in top_words_v2:
    print(f"{word}: {score : .3f}")


Top 20 most distinctive words (Without stopwords) : 
like:  649.699
just:  597.158
love:  483.638
don:  448.206
good:  393.803
thanks:  353.434
people:  352.502
thank:  343.372
know:  332.395
think:  319.482
really:  318.588
lol:  313.553
oh:  290.097
time:  282.165
ve:  253.935
did:  239.282
got:  232.921
right:  232.779
yeah:  225.105
ll:  216.184


In [8]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf_v2 , Y_train)

print("Model trained successfully")

Model trained successfully


In [9]:
Y_pred = model.predict(X_test_tfidf_v2)

for i in range (10):
    text=X_test[i]
    true_emotion = label_names[Y_test[i]]
    predicted_emotion = label_names[Y_pred[i]]
    match =  "✓" if true_emotion==predicted_emotion else "✗"
    print(f"{match} Text : {text}")
    print(f"  true : {true_emotion} | Predicted : {predicted_emotion} ")
    print("----")

✗ Text : I’m really sorry about your situation :( Although I love the names Sapphira, Cirilla, and Scarlett!
  true : sadness | Predicted : love 
----
✗ Text : It's wonderful because it's awful. At not with.
  true : admiration | Predicted : disgust 
----
✗ Text : Kings fan here, good luck to you guys! Will be an interesting game to watch! 
  true : excitement | Predicted : admiration 
----
✓ Text : I didn't know that, thank you for teaching me something today!
  true : gratitude | Predicted : gratitude 
----
✓ Text : They got bored from haunting earth for thousands of years and ultimately moved on to the afterlife.
  true : neutral | Predicted : neutral 
----
✓ Text : Thank you for asking questions and recognizing that there may be things that you don’t know or understand about police tactics. Seriously. Thank you.
  true : gratitude | Predicted : gratitude 
----
✓ Text : You’re welcome
  true : gratitude | Predicted : gratitude 
----
✗ Text : 100%! Congrats on your job too!
  true : 

In [10]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(Y_test, Y_pred)

print(f"Accuracy : {accuracy : .2%}")

Accuracy :  54.92%


In [18]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(
    Y_test,
    Y_pred,
    target_names = label_names,
    output_dict=True,
    zero_division=0    
)

df = pd.DataFrame(report).transpose()
df_sorted =  df.sort_values(by='f1-score', ascending=True)

print(df_sorted)


#print(report)



TypeError: DataFrame.sort_values() got an unexpected keyword argument 'descending'. Did you mean 'ascending'?